# Notebook 05 — Training a Tiny GPT

**Goal:** Wire up the full model, train it on a small corpus, and generate text.

---

### Contents
1. [Configuration](#1)
2. [Prepare the corpus](#2)
3. [Train the model](#3)
4. [Generate text](#4)
5. [Key takeaways](#5)


In [6]:
# ── Imports ──────────────────────────────────────────────────────────────────
import os, sys, time, math
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import tiktoken
from tqdm.auto import tqdm

sys.path.insert(0, os.path.abspath('.'))
from utils.model import GPTModel

# ── Reproducibility ───────────────────────────────────────────────────────────
torch.manual_seed(42)
np.random.seed(42)
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

# ── Device ────────────────────────────────────────────────────────────────────
device = torch.device(
    'cuda' if torch.cuda.is_available() else
    'mps'  if torch.backends.mps.is_available() else 'cpu'
)

print('=' * 60)
print('  Notebook 05 — Training a Tiny GPT')
print('=' * 60)
print(f'  Device : {device}')
print('=' * 60)


  Notebook 05 — Training a Tiny GPT
  Device : mps


<a id='1'></a>
## 1 — Configuration

Everything is in one dict — change these values to scale up later.


In [7]:
config = {
    # ── Model architecture ────────────────────────────────────────────────────
    'vocab_size':   50257,    # GPT-2 tokenizer
    'd_model':      64,       # embedding / hidden dimension
    'n_heads':      4,        # attention heads
    'n_layers':     2,        # transformer blocks
    'd_ff':         256,      # FFN inner dimension (4 × d_model)
    'max_seq_len':  64,       # context window
    'dropout':      0.1,

    # ── Training ──────────────────────────────────────────────────────────────
    'batch_size':     16,
    'learning_rate':  3e-4,
    'num_steps':      300,
    'eval_interval':  50,
}

for k, v in config.items():
    print(f'  {k:>14}: {v}')


      vocab_size: 50257
         d_model: 64
         n_heads: 4
        n_layers: 2
            d_ff: 256
     max_seq_len: 64
         dropout: 0.1
      batch_size: 16
   learning_rate: 0.0003
       num_steps: 300
   eval_interval: 50


<a id='2'></a>
## 2 — Prepare the corpus

A small engineering/science text — enough to demonstrate the pipeline, not enough
for high-quality generation.


In [8]:
corpus = """
Engineering is the application of scientific principles to design and build systems.
A control system measures the behaviour of a process and compares it with a desired target.
Embedded systems are designed to perform specific functions within larger machines.
Machine learning enables models to improve from data rather than explicit programming.
A transformer uses self attention to process sequences in parallel.
Functional safety is concerned with correct behaviour in response to inputs and faults.
Robotics combines sensors actuators and control to achieve useful physical behaviour.
Software engineering uses testing version control and automation to build reliable systems.
"""

enc  = tiktoken.get_encoding('gpt2')
data = torch.tensor(enc.encode(corpus), dtype=torch.long)

split      = int(0.9 * len(data))
train_data = data[:split]
val_data   = data[split:]

print(f'Total tokens      : {len(data)}')
print(f'Training tokens   : {len(train_data)}')
print(f'Validation tokens : {len(val_data)}')


Total tokens      : 119
Training tokens   : 107
Validation tokens : 12


In [9]:
def get_batch(data, batch_size, seq_len, device):
    """Random (input, target) pairs — target is input shifted right by 1."""
    starts = torch.randint(0, len(data) - seq_len - 1, (batch_size,))
    x = torch.stack([data[i : i + seq_len]     for i in starts])
    y = torch.stack([data[i + 1 : i + seq_len + 1] for i in starts])
    return x.to(device), y.to(device)


# Quick sanity check
x, y = get_batch(train_data, 2, 8, device)
print(f'Input  : {tuple(x.shape)}')
print(f'Target : {tuple(y.shape)}')


Input  : (2, 8)
Target : (2, 8)


<a id='3'></a>
## 3 — Train the model


In [10]:
model     = GPTModel(config).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=config['learning_rate'])

print(f'Trainable parameters: {model.count_parameters():,}')

train_losses = []
val_losses   = []
val_steps    = []


@torch.no_grad()
def estimate_loss(split_data, n_batches=10):
    model.eval()
    losses = [
        F.cross_entropy(
            model(xb := get_batch(split_data, config['batch_size'],
                                  config['max_seq_len'], device)[0]).view(-1, config['vocab_size']),
            get_batch(split_data, config['batch_size'],
                      config['max_seq_len'], device)[1].view(-1),
        ).item()
        for _ in range(n_batches)
    ]
    model.train()
    return sum(losses) / len(losses)


start = time.time()
model.train()

for step in tqdm(range(config['num_steps']), desc='Training'):
    xb, yb = get_batch(train_data, config['batch_size'], config['max_seq_len'], device)
    logits  = model(xb)
    loss    = F.cross_entropy(logits.view(-1, config['vocab_size']), yb.view(-1))

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()

    train_losses.append(loss.item())

    if step % config['eval_interval'] == 0:
        vl = estimate_loss(val_data, n_batches=5)
        val_losses.append(vl)
        val_steps.append(step)
        print(f'  step {step:>3d} | train {loss.item():.4f} | val {vl:.4f}')

elapsed = time.time() - start
print(f'\nDone in {elapsed:.1f}s  —  final train loss: {train_losses[-1]:.4f}')


Trainable parameters: 3,316,032


Training:   0%|          | 0/300 [00:00<?, ?it/s]

RuntimeError: random_ expects 'from' to be less than 'to', but got from=0 >= to=-53

In [ ]:
# ── Loss curve ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(train_losses, alpha=0.6, label='train')
ax.plot(val_steps, val_losses, 'o-', label='val')
ax.set_xlabel('Step')
ax.set_ylabel('Cross-entropy loss')
ax.set_title('Training curve', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# ── Save checkpoint for notebook 06 ───────────────────────────────────────────
os.makedirs('checkpoints', exist_ok=True)
torch.save({
    'model_state_dict': model.state_dict(),
    'config': config,
}, 'checkpoints/model.pt')
print('Saved → checkpoints/model.pt')


<a id='4'></a>
## 4 — Generate text


In [ ]:
def generate(prompt, max_new=40, temperature=0.8, top_k=20):
    ids = torch.tensor([enc.encode(prompt)], dtype=torch.long, device=device)
    out = model.generate(ids, max_new_tokens=max_new,
                         temperature=temperature, top_k=top_k)
    return enc.decode(out[0].tolist())


for prompt in ['Engineering is', 'A transformer', 'Functional safety']:
    print('=' * 60)
    print(f'PROMPT: {prompt!r}')
    print(generate(prompt))
    print()

print('(Tiny model + tiny corpus → limited quality.  Scale up the config to improve.)')


<a id='5'></a>
## 5 — Key takeaways

- The full model is: **embedding → positional encoding → N × TransformerBlock → LN → linear head**
- Loss function: cross-entropy over the vocabulary (next-token prediction)
- Optimizer: AdamW with gradient clipping
- The `config` dict controls every dimension — change it to scale up
- The saved checkpoint is loaded by Notebook 06 for attention visualisation
